In [ ]:
from pathlib import Path
import importlib
import re

import numpy as np
import matplotlib.pyplot as plts
import yaml

# Reload local analysis modules so rerunning this notebook in a long-lived
# kernel uses the current function signatures rather than cached imports.
import pmt.preprocessing as _pmt_preprocessing
import pmt.selection as _pmt_selection
import pmt.io as _pmt_io
importlib.reload(_pmt_preprocessing)
importlib.reload(_pmt_selection)
importlib.reload(_pmt_io)

from pmt.io import *
from lab_tools.io import read_keysight_h5_direct, standard_units
from scipy.signal import find_peaks

from pmt.preprocessing import *
from pmt.config import *
from pmt.plotting import *
from pmt.selection import *
from pmt.fit import *
from pmt.fit.core import *

from pmt.fit.poisson import fit_poisson_spe, poisson_npe_fractions
from pmt.fit.bellamy import fit_bellamy_spe

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.60,
})


# Select Files


In [ ]:
# file_name = "WA0089_750V_525MHz_led100" 
# file_name = "WA0089_775V_525MHz_led100" 
# file_name = "WA0089_800V_525MHz_led100" 
file_name = globals().get("batch_file_name", "WA0089_825V_525MHz_led100")
voltage_match = re.search(r"(?:^|_)(\d+(?:\.\d+)?)V(?:_|$)", file_name)
if voltage_match is None:
    raise ValueError(f"Could not determine PMT voltage from file name: {file_name}")
voltage_V = float(voltage_match.group(1))
# file_name = "WA0089_850V_525MHz_led100"
# file_name = "WA0089_875V_525MHz_led100" 
# file_name = "WA0089_900V_525MHz_led100"
# file_name = "WA0089_925V_525MHz_led100"
# file_name = "WA0089_950V_525MHz_led100"
# file_name = "WA0089_975V_525MHz_led100" 
# file_name = "WA0089_1000V_525MHz_led100" 

channel = globals().get("batch_channel", "Channel 3")
data_dir = Path(globals().get("batch_data_dir", "data/WA0089/voltage_sweep/260706"))
# Set to a number for quick iteration, or None to use all matching files.
max_files = None
baseline_window_ns = (0, 20)

# Must match the trigger-relative integration settings used in Selection.ipynb.
led_time_ns = 36.0
pre_led_ns = 20.0
post_led_ns = 80.0
peak_snr_threshold = float(globals().get("batch_peak_snr_threshold", 5.0))
peak_prominence_snr = globals().get("batch_peak_prominence_snr", None)
peak_distance_samples = globals().get("batch_peak_distance_samples", None)
peak_width_samples = globals().get("batch_peak_width_samples", None)

# save
save_plots = globals().get("batch_save_plots", False)
fit_output_root = Path(globals().get("batch_fit_output_dir", Path('plots/260706/fit')))
# Keep plots grouped by voltage while leaving machine-readable fit results
# in the shared fit_results directory used by Gain_calibration.ipynb.
save_dir = str(fit_output_root / f"{voltage_V:g}V")
file_nickname = file_name
fit_inputs = str(globals().get("batch_fit_inputs_dir", Path("fit_data")))

save_fit_res = "fit_results"
savefit_path = fit_output_root / save_fit_res
savefit_path.mkdir(parents=True, exist_ok=True)


In [ ]:

# Load the dataframe before the optional pulse-shape/timing cuts. This is
# important: *_df_selected.pkl already has those cuts baked in and therefore
# cannot be used for a cut/no-cut comparison.
df_file = Path(fit_inputs) / f"{file_nickname}_df.pkl"
require_cached_fit_data = bool(globals().get("batch_require_cached_fit_data", False))
baseline_reference_spec = globals().get("batch_baseline_reference_path", None)
baseline_reference_path = resolve_baseline_reference_path(
    baseline_reference_spec, file_name
)
baseline_reference_time_ns = None
baseline_reference_mV = None
if baseline_reference_path is not None:
    reference_data = load_baseline_reference(baseline_reference_path)
    baseline_reference_time_ns = reference_data["time_ns"]
    baseline_reference_mV = reference_data["baseline_template_mV"]
    print(f"Using median baseline reference from {baseline_reference_path}")

if df_file.exists():
    print(f"Loading cached pre-cut dataframe from {df_file}")
    df = pd.read_pickle(df_file)
elif require_cached_fit_data:
    raise FileNotFoundError(
        f"Missing cached dataframe {df_file}. Set run_selection=True first, "
        "or set batch_require_cached_fit_data=False to allow raw HDF5 loading."
    )
else:
    print("Creating dataframe from raw waveforms...")
    chunk_size = 512
    baseline_window_ns = (0, 20)
    files = find_pmt_files(data_dir, file_name, max_files=max_files)
    for file in files:
        print(file)
    print(f"Number of files loaded: {len(files)}")
    time_ns, waveforms_mV, df = load_files_one_go(
        files, chunk_size, baseline_window_ns, channel,
        led_time_ns=led_time_ns, pre_led_ns=pre_led_ns, post_led_ns=post_led_ns,
        peak_snr_threshold=peak_snr_threshold,
        peak_prominence_snr=peak_prominence_snr,
        peak_distance_samples=peak_distance_samples,
        peak_width_samples=peak_width_samples,
        baseline_reference_time_ns=baseline_reference_time_ns,
        baseline_reference_mV=baseline_reference_mV,
    )

required_charge_columns = {"area_mV_ns", "charge_led_window_mV_ns"}
missing_charge_columns = required_charge_columns.difference(df.columns)
if missing_charge_columns:
    raise ValueError(
        f"Missing charge columns {sorted(missing_charge_columns)}. "
        "Rerun Selection.ipynb to regenerate the pre-cut dataframe with "
        "the fixed LED integration window."
    )
if baseline_reference_path is not None and "baseline_reference_residual_mean_mV" not in df.columns:
    raise ValueError(
        "Cached fit data were created without median-reference baseline subtraction. "
        "Rerun Batch_analysis.ipynb with run_selection=True."
    )

print(df.shape)


In [ ]:

# Use per-waveform SNR so the threshold follows the measured baseline noise.
# The standard selection was tuned for fast, unfiltered pulses. The 20 MHz
# bandwidth-limited data can use looser modes that do not require one sharp peak.
timing_reference_snr = float(globals().get("batch_timing_reference_snr", 30.0))
cut_thresholds_snr = [float(x) for x in globals().get("batch_cut_thresholds_snr", [15.0, 20.0, 25.0, 30.0, 40.0])]
peak_timing_tolerance_ns = float(globals().get("batch_peak_timing_tolerance_ns", 5.0))
selection_mode = globals().get("batch_selection_mode", "standard")
timing_reference_requires_single_peak = bool(globals().get(
    "batch_timing_reference_requires_single_peak",
    selection_mode == "standard",
))
include_no_peak_cuts = bool(globals().get("batch_include_no_peak_cuts", True))
include_shape_cut = bool(globals().get("batch_include_shape_cut", True))
max_allowed_peaks = int(globals().get("batch_max_allowed_peaks", 6))
selection_name_filter = globals().get("batch_selection_names", None)
if selection_name_filter is not None:
    selection_name_filter = set(selection_name_filter)
    # A filtered selection name is authoritative. Infer its mode so a stale
    # batch_selection_mode left in an interactive kernel cannot make a valid
    # name disappear from the generated configurations.
    requested_modes = set()
    for requested_name in selection_name_filter:
        if requested_name == "no_peak_cuts":
            continue
        if requested_name.startswith("led_timing_above_snr"):
            requested_modes.add("timing_only")
        elif requested_name.startswith("loose_peak_timing_above_snr"):
            requested_modes.add("loose_peak_multiplicity")
        elif requested_name.startswith("pulse_quality_above_snr"):
            requested_modes.add("standard")
        elif requested_name.startswith("dark_count_quality_above_snr"):
            requested_modes.add("dark_counts")
    if len(requested_modes) > 1:
        raise ValueError(
            "batch_selection_names mixes selection modes; choose names from "
            "only one of led_timing, loose_peak_timing, pulse_quality, or dark_count_quality."
        )
    if requested_modes:
        inferred_selection_mode = requested_modes.pop()
        if selection_mode != inferred_selection_mode:
            print(
                f"Inferring batch_selection_mode={inferred_selection_mode!r} "
                f"from batch_selection_names (was {selection_mode!r})."
            )
            selection_mode = inferred_selection_mode
        if "batch_timing_reference_requires_single_peak" not in globals():
            timing_reference_requires_single_peak = selection_mode == "standard"


valid_selection_modes = {"standard", "timing_only", "loose_peak_multiplicity", "dark_counts"}
if selection_mode not in valid_selection_modes:
    raise ValueError(
        f"Unknown batch_selection_mode={selection_mode!r}. "
        f"Choose one of {sorted(valid_selection_modes)}."
    )

is_single_peak = df["n_peaks"] == 1
is_loose_peak_multiplicity = df["n_peaks"].between(1, max_allowed_peaks)

timing_reference_mask = (
    (df["snr"] >= timing_reference_snr)
    & np.isfinite(df["peak_time_ns"])
)
if timing_reference_requires_single_peak:
    timing_reference_mask &= is_single_peak

timing_reference = df.loc[timing_reference_mask, "peak_time_ns"]
uses_led_timing = selection_mode != "dark_counts"
allowed_peak_window_ns = None
expected_peak_time_ns = np.nan
if uses_led_timing:
    if timing_reference.empty:
        raise ValueError("No clean events are available to estimate the LED peak time.")
    timing_bin_width_ns = 1.0
    timing_bins = np.arange(
        timing_reference.min(), timing_reference.max() + timing_bin_width_ns, timing_bin_width_ns
    )
    timing_counts, timing_edges = np.histogram(timing_reference, bins=timing_bins)
    peak_bin = np.argmax(timing_counts)
    expected_peak_time_ns = 0.5 * (timing_edges[peak_bin] + timing_edges[peak_bin + 1])
    allowed_peak_window_ns = (
        expected_peak_time_ns - peak_timing_tolerance_ns,
        expected_peak_time_ns + peak_timing_tolerance_ns,
    )
    is_led_aligned = df["peak_time_ns"].between(*allowed_peak_window_ns)
else:
    is_led_aligned = np.ones(len(df), dtype=bool)
    print("Dark-count mode: LED timing estimation and timing cuts are disabled.")

# Derive loose central-98% pulse-shape ranges from clean LED pulses when requested.
shape_columns = [
    "peak_width_ns",
    "rise_time_10_90_ns",
    "fall_time_90_10_ns",
]
shape_reference_mask = timing_reference_mask & np.isfinite(df[shape_columns]).all(axis=1)
if uses_led_timing:
    shape_reference_mask &= is_led_aligned
shape_reference = df.loc[shape_reference_mask, shape_columns]
min_shape_reference_pulses = 100
shape_cut_available = len(shape_reference) >= min_shape_reference_pulses
shape_cut_ranges = {}
if include_shape_cut and shape_cut_available:
    shape_quantiles = (0.01, 0.99)
    shape_cut_ranges = {
        column: tuple(shape_reference[column].quantile(shape_quantiles))
        for column in shape_columns
    }
elif include_shape_cut:
    print(
        f"Warning: only {len(shape_reference)} clean pulses are available; "
        f"at least {min_shape_reference_pulses} are required to derive stable "
        "pulse-shape ranges. The pulse-shape cut will be skipped."
    )
is_pulse_shaped = np.ones(len(df), dtype=bool)
for column, limits in shape_cut_ranges.items():
    is_pulse_shaped &= df[column].between(*limits).to_numpy()
    print(f"{column}: {limits[0]:.3g} to {limits[1]:.3g} ns")

selection_configs = []
if include_no_peak_cuts:
    selection_configs.append({
        "name": "no_peak_cuts",
        "label": "No peak cuts",
        "cuts": [],
    })

for threshold_snr in cut_thresholds_snr:
    threshold_tag = f"snr{threshold_snr:g}"
    # A missing SNR means no qualifying peak; preserve those pedestal-like
    # events just like events below the cut threshold.
    is_at_or_below_threshold = (
        df["snr"].isna() | (df["snr"] < threshold_snr)
    )

    if selection_mode == "standard":
        selection_name = f"pulse_quality_above_{threshold_tag}"
        selection_label = "Single peak + LED timing"
        quality_cuts = [
            (
                f"SNR >= {threshold_snr:g}: single peak",
                is_at_or_below_threshold | is_single_peak,
            ),
            (
                f"SNR >= {threshold_snr:g}: LED peak timing",
                is_at_or_below_threshold | is_led_aligned,
            ),
        ]
    elif selection_mode == "dark_counts":
        selection_name = f"dark_count_quality_above_{threshold_tag}"
        selection_label = "Single peak (timing ignored)"
        quality_cuts = [
            (
                f"SNR >= {threshold_snr:g}: single peak",
                is_at_or_below_threshold | is_single_peak,
            ),
        ]
    elif selection_mode == "timing_only":
        selection_name = f"led_timing_above_{threshold_tag}"
        selection_label = "LED timing"
        quality_cuts = [
            (
                f"SNR >= {threshold_snr:g}: LED peak timing",
                is_at_or_below_threshold | is_led_aligned,
            ),
        ]
    else:
        selection_name = f"loose_peak_timing_above_{threshold_tag}"
        selection_label = f"1-{max_allowed_peaks} peaks + LED timing"
        quality_cuts = [
            (
                f"SNR >= {threshold_snr:g}: 1-{max_allowed_peaks} peaks",
                is_at_or_below_threshold | is_loose_peak_multiplicity,
            ),
            (
                f"SNR >= {threshold_snr:g}: LED peak timing",
                is_at_or_below_threshold | is_led_aligned,
            ),
        ]

    if include_shape_cut and shape_cut_available:
        quality_cuts.append((
            f"SNR >= {threshold_snr:g}: pulse shape",
            is_at_or_below_threshold | is_pulse_shaped,
        ))
    if include_shape_cut:
        shape_label = " + shape" if shape_cut_available else " (shape cut skipped)"
    else:
        shape_label = ""
    selection_configs.append({
        "name": selection_name,
        "label": f"{selection_label}{shape_label} at SNR >= {threshold_snr:g}",
        "cuts": quality_cuts,
    })

if selection_name_filter is not None:
    available_selection_names = {config["name"] for config in selection_configs}
    unknown_selection_names = selection_name_filter - available_selection_names
    if unknown_selection_names:
        raise ValueError(f"Unknown batch selection names: {sorted(unknown_selection_names)}")
    selection_configs = [
        config for config in selection_configs
        if config["name"] in selection_name_filter
    ]
if not selection_configs:
    raise ValueError("No selection configurations are enabled.")

selected_dfs = {}
selection_rejected_masks = {}
for config in selection_configs:
    print(f"\n--- {config['label']} ---")
    cutflow = CutFlow(len(df))
    for cut_name, keep_mask in config["cuts"]:
        cutflow.apply(cut_name, keep_mask)
    cutflow.print()
    selected_dfs[config["name"]] = df.loc[cutflow.mask].copy()
    selection_rejected_masks[config["name"]] = ~cutflow.mask

print(f"Selection mode: {selection_mode}")
print(f"Timing reference: SNR >= {timing_reference_snr:g}")
print(f"Timing reference requires single peak: {timing_reference_requires_single_peak}")
print(f"Expected LED peak time: {expected_peak_time_ns:.2f} ns")
print(f"Allowed peak window: {allowed_peak_window_ns} ns")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].hist(df["peak_time_ns"], bins=200, histtype="step", label="All events")
axes[0].hist(timing_reference, bins=200, histtype="step", label="Timing reference")
axes[0].axvline(expected_peak_time_ns, color="tab:red", label="Expected LED peak")
if uses_led_timing:
    axes[0].axvspan(*allowed_peak_window_ns, color="tab:green", alpha=0.2, label="Accepted window")
axes[0].set(xlabel="Peak time [ns]", ylabel="Events", yscale="log")
axes[0].legend()

for config in selection_configs:
    values = selected_dfs[config["name"]]["area_mV_ns"]
    axes[1].hist(
        values, bins=nbins if "nbins" in globals() else 250,
        density=True, histtype="step",
        label=f"{config['label']} (N={len(values):,})",
    )
axes[1].set(xlabel="Charge [mV ns]", ylabel="Density", yscale="log")
axes[1].legend(fontsize="small")
fig.tight_layout()
save_plot(fig, save_plots, save_dir, file_nickname, "selection_comparison", Nevents=None)


In [ ]:
# Show full-record examples of the two populations underlying calibration.
# n_peaks == 0 is treated as pedestal; signal examples must have exactly one
# detected peak, matching the later quality cut, and exceed the configurable SNR.
n_example_waveforms = int(globals().get("batch_example_waveform_count", 10))
example_high_snr_threshold = float(globals().get("batch_example_high_snr_threshold", 15.0))
example_waveform_seed = int(globals().get("batch_example_waveform_seed", 12345))
example_rng = np.random.default_rng(example_waveform_seed)

example_populations = [
    (
        "pedestal_no_detected_peak",
        df["n_peaks"].eq(0),
        "Assumed pedestal (no detected peak)",
        "tab:blue",
    ),
    (
        "single_peak_high_snr",
        df["n_peaks"].eq(1) & df["snr"].ge(example_high_snr_threshold),
        f"High-SNR single peak (SNR >= {example_high_snr_threshold:g})",
        "tab:red",
    ),
]

fig, ax = plt.subplots(figsize=(15, 6))
plotted_population_count = 0
time_limits = None

for population_name, population_mask, population_label, population_color in example_populations:
    population_df = df.loc[population_mask]
    sample_size = min(n_example_waveforms, len(population_df))
    if sample_size == 0:
        print(f"No events available for waveform diagnostic: {population_label}")
        continue
    sample_indices = example_rng.choice(
        population_df.index.to_numpy(), size=sample_size, replace=False
    )
    example_df = population_df.loc[sample_indices].sort_index()
    example_time_ns, example_waveforms = load_event_waveforms(
        example_df,
        channel=channel,
        baseline_window_ns=baseline_window_ns,
        baseline_reference_time_ns=baseline_reference_time_ns,
        baseline_reference_mV=baseline_reference_mV,
    )

    time_limits = (example_time_ns[0], example_time_ns[-1])
    for waveform_index, waveform in enumerate(example_waveforms):
        ax.plot(
            example_time_ns, waveform, color=population_color,
            linewidth=1.0, alpha=0.58,
            label=(f"{population_label} (N={sample_size})" if waveform_index == 0 else None),
        )
    plotted_population_count += 1

if plotted_population_count:
    ax.axvspan(*baseline_window_ns, color="tab:orange", alpha=0.14, label="Baseline window")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.45)
    ax.set(
        xlabel="Time [ns]", ylabel="Voltage [mV]",
        title="Assumed pedestal and high-SNR signal waveforms — common voltage scale",
        xlim=time_limits,
    )
    ax.legend()
    fig.tight_layout()
    save_plot(
        fig, save_plots, save_dir, file_nickname,
        "example_waveforms_pedestal_vs_high_snr", Nevents=None,
    )
else:
    plt.close(fig)


In [ ]:

# Optional diagnostic: reload a small sample of rejected waveforms for each
# active selection. This uses cached event_file/event_segment bookkeeping and
# reads only the requested events, not the full HDF5 dataset.
plot_rejected_waveforms = bool(globals().get("batch_plot_rejected_waveforms", False))
n_rejected_waveforms = int(globals().get("batch_rejected_waveform_sample", 40))

if plot_rejected_waveforms:
    rng = np.random.default_rng(int(globals().get("batch_rejected_waveform_seed", 12345)))
    for config in selection_configs:
        name = config["name"]
        if name == "no_peak_cuts":
            continue
        # Record every failed cut independently; one event may have several reasons.
        failed_cuts_by_event = {index: [] for index in df.index}
        for cut_name, keep_mask in config["cuts"]:
            keep_by_index = pd.Series(np.asarray(keep_mask, dtype=bool), index=df.index)
            for index in keep_by_index.index[~keep_by_index]:
                failed_cuts_by_event[index].append(cut_name)
        rejected_indices = [
            index for index, reasons in failed_cuts_by_event.items() if reasons
        ]
        rejected_df = df.loc[rejected_indices].copy()
        rejected_df["failed_cuts"] = [
            "; ".join(failed_cuts_by_event[index]) for index in rejected_indices
        ]
        if rejected_df.empty:
            print(f"No rejected events for {name}")
            continue

        sample_size = min(n_rejected_waveforms, len(rejected_df))
        sample_indices = rng.choice(rejected_df.index.to_numpy(), size=sample_size, replace=False)
        rejected_sample = rejected_df.loc[sample_indices].sort_index()
        rejected_time_ns, rejected_waveforms = load_event_waveforms(
            rejected_sample,
            channel=channel,
            baseline_window_ns=baseline_window_ns,
            baseline_reference_time_ns=globals().get("baseline_reference_time_ns", None),
            baseline_reference_mV=globals().get("baseline_reference_mV", None),
        )

        # Rerun the same detector so the diagnostic can show every peak that
        # contributed to cached n_peaks. The cache must have been regenerated
        # after changing these settings for the two counts to agree.
        _, detected_peaks, detected_properties = _pmt_preprocessing._find_waveform_peaks(
            rejected_waveforms,
            rejected_sample["baseline_rms_mV"].to_numpy(),
            polarity="negative",
            peak_snr_threshold=peak_snr_threshold,
            peak_prominence_snr=peak_prominence_snr,
            peak_distance_samples=peak_distance_samples,
            peak_width_samples=peak_width_samples,
        )

        # Overlay all sampled waveforms. Color identifies the failed cut(s);
        # each x marker and annotation identifies one detector-accepted peak.
        rejection_reasons = rejected_sample["failed_cuts"].unique().tolist()
        color_map = plt.get_cmap("tab10")
        reason_colors = {
            reason: color_map(i % color_map.N)
            for i, reason in enumerate(rejection_reasons)
        }
        reasons_already_labeled = set()
        fig, ax = plt.subplots(figsize=(15, 6))
        for trace_number, ((event_index, event), waveform, peaks, properties) in enumerate(zip(
            rejected_sample.iterrows(), rejected_waveforms, detected_peaks, detected_properties
        ), start=1):
            reason = event["failed_cuts"]
            detected_count = len(peaks)
            cached_count = int(event["n_peaks"])
            count_text = f"n_peaks={cached_count}"
            if detected_count != cached_count:
                count_text += f" (current settings: {detected_count})"
            label = (
                f"trace {trace_number}, event {event_index}: {reason}; {count_text}"
                if reason not in reasons_already_labeled
                else f"trace {trace_number}, event {event_index}: {count_text}"
            )
            ax.plot(
                rejected_time_ns, waveform, color=reason_colors[reason],
                linewidth=0.9, alpha=0.65, label=label,
            )
            baseline_rms = float(event["baseline_rms_mV"])
            peak_heights = properties.get("peak_heights", np.full(detected_count, np.nan))
            prominences = properties.get("prominences", np.full(detected_count, np.nan))
            widths = properties.get("widths", np.full(detected_count, np.nan))
            dt_ns = float(np.mean(np.diff(rejected_time_ns)))
            for peak_number, (peak, height, prominence, width) in enumerate(
                zip(peaks, peak_heights, prominences, widths), start=1
            ):
                peak_time = rejected_time_ns[peak]
                peak_voltage = waveform[peak]
                ax.scatter(peak_time, peak_voltage, marker="x", s=55, linewidths=1.5,
                           color=reason_colors[reason], zorder=5)
                ax.annotate(
                    (f"T{trace_number} P{peak_number}: {peak_time:.2f} ns\n"
                     f"height={height:.2f} mV ({height / baseline_rms:.1f} RMS)\n"
                     f"prom={prominence:.2f} mV ({prominence / baseline_rms:.1f} RMS), "
                     f"width={width * dt_ns:.2f} ns"),
                    xy=(peak_time, peak_voltage), xytext=(5, -10 - 35 * (peak_number - 1)),
                    textcoords="offset points", fontsize=7, color=reason_colors[reason],
                    arrowprops={"arrowstyle": "-", "color": reason_colors[reason], "lw": 0.7},
                )
            reasons_already_labeled.add(reason)
        ax.axvspan(*baseline_window_ns, color="tab:orange", alpha=0.18, label="Baseline window")
        if uses_led_timing:
            ax.axvspan(*allowed_peak_window_ns, color="tab:green", alpha=0.14, label="Accepted peak-time window")
        ax.set(
            xlabel="Time [ns]", ylabel="Voltage [mV]",
            title=f"Waveforms rejected by {name} (N={sample_size})",
        )
        ax.legend(title="Rejected by", fontsize=8, title_fontsize=9)
        fig.tight_layout()
        save_plot(
            fig, save_plots, save_dir, file_nickname,
            f"rejected_waveforms_{name}", Nevents=None,
        )
else:
    print("Rejected-waveform diagnostic disabled")


# Fit one step

In [ ]:

# Compare the original full-record integral with a trigger-relative LED
# window. Both remain defined for pedestal events.
all_charge_methods = {
    "full_waveform": {"column": "area_mV_ns", "label": "Full waveform"},
    "led_window": {
        "column": "charge_led_window_mV_ns",
        "label": f"LED window [-{pre_led_ns:g}, +{post_led_ns:g}] ns",
    },
}
charge_method_filter = globals().get("batch_charge_methods", list(all_charge_methods))
charge_method_filter = list(charge_method_filter)
unknown_charge_methods = set(charge_method_filter) - set(all_charge_methods)
if unknown_charge_methods:
    raise ValueError(f"Unknown batch charge methods: {sorted(unknown_charge_methods)}")
charge_methods = {
    name: all_charge_methods[name]
    for name in charge_method_filter
}
if not charge_methods:
    raise ValueError("No charge methods are enabled.")
fit_models = {model.lower() for model in globals().get("batch_fit_models", ["poisson", "bellamy"])}
unknown_fit_models = fit_models - {"poisson", "bellamy"}
if unknown_fit_models:
    raise ValueError(f"Unknown batch fit models: {sorted(unknown_fit_models)}")
if not fit_models:
    raise ValueError("No fit models are enabled.")
print(f"Enabled charge methods: {list(charge_methods)}")
print(f"Enabled fit models: {sorted(fit_models)}")

fit_samples = {}
fit_ranges = {}
for method, method_config in charge_methods.items():
    charge_column = method_config["column"]
    fit_samples[method] = {}
    print(f"\n--- {method_config['label']} ({charge_column}) ---")
    for config in selection_configs:
        name = config["name"]
        values = selected_dfs[name][charge_column].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            raise ValueError(f"No finite {charge_column} values for selection {name}")
        fit_samples[method][name] = values
        print(f"{config['label']}: {len(values):,} finite events")

    all_charge = np.concatenate(list(fit_samples[method].values()))
    fit_ranges[method] = (all_charge.min(), all_charge.max())
    print(f"Range = ({fit_ranges[method][0]:.2f}, {fit_ranges[method][1]:.2f})")


# Fit Config

optional: put all the initial parameter values and limits into a separate file (yaml format)

In [ ]:
# with open('config_files/fit_config.yaml', 'r') as file:
#     cfg = yaml.safe_load(file)[file_nickname]
#     try:
#         fit_rg = cfg['fit_range']
#     except:
#         print('No fit range specified, using full range')
#         fit_rg = fit_range_full

# print(f'Fit range = {fit_rg}')

# p0_poisson = cfg['poisson']['initPars']
# p0_bellamy = cfg['bellamy']['initPars']


# Batch_analysis.ipynb can override the maximum modeled NPE. Standalone runs
# retain the historical default of 3.
maxPE = globals().get("batch_max_npe", 3)
if not isinstance(maxPE, int) or isinstance(maxPE, bool) or maxPE < 0:
    raise ValueError("batch_max_npe must be a non-negative integer")
nbins = 250

## Poisson

In [ ]:
## if you don't want to use the config file:
# [mean value, lower bound, upper bound, is_fixed]
p0_poisson = {
    'q0_mV_ns': [-0.6, -6.0, 10.0, False],
    'sigma0_mV_ns': [3.9, '1e-4', 50.0, False],
    'q1_mV_ns': [29.0, 0.5, 200.0, False],
    'sigma1_mV_ns': [12.0, 0.2, 100.0, False],
    'mu_pe': [0.06, 0.001, 5.0, False]
}
# Each integration method uses the range calculated from its own samples.

In [ ]:

from copy import deepcopy

fit_results = {
    method: {config["name"]: {} for config in selection_configs}
    for method in charge_methods
}

if "poisson" in fit_models:
  for method, method_config in charge_methods.items():
    for config in selection_configs:
      name, label = config["name"], config["label"]
      charge_mV_ns = fit_samples[method][name]
      p0 = deepcopy(p0_poisson)
      p0["n_total"] = [len(charge_mV_ns), 0.5 * len(charge_mV_ns), 1.5 * len(charge_mV_ns), False]

      print(f"\n===== Poisson: {method_config['label']} - {label} =====")
      result = fit_poisson_spe(
          charge_mV_ns, p0=p0, max_pe=maxPE, bins=nbins, fit_range=fit_ranges[method]
      )
      fit_results[method][name]["poisson"] = result
      print_fit_result_table(result)
      print(result["diagnostics"])

      title = f"Poisson — {method_config['label']} — {label} — {voltage_V:g} V"
      fig, ax_fit, ax_resid, ax_corr = plot_fit_summary(
          result, title=title, shrink_colorbar=0.7,
          component_visibility_fraction=0, logscale=True,
          show_event_fractions=True, max_fraction_pe=maxPE,
      )
      save_plot(fig, save_plots, save_dir, file_nickname,
                f"fit_Poisson_{method}_{name}_{nbins}bins", Nevents=None)
      save_fit_results(result, output_path=f"{savefit_path}/{file_nickname}_{method}_{name}_poisson.yaml")
else:
    print("Skipping Poisson fits")


## Bellamy

In [ ]:
p0_bellamy = {
    'q0_mV_ns': [-0.6, -6.0, 10.0, False],
    'sigma0_mV_ns': [3.9, '1e-4', 50.0, False],
    'q1_mV_ns': [30.0, 0.5, 200.0, False],
    'sigma1_mV_ns': [12.0, 0.2, 100.0, False],
    'mu_pe': [0.06, 0.001, 5.0, False],
    'w': [0.2, 0.0, 1.0, False],
    'alpha': [0.3, 0.0, 10.0, False]
 }
# Each integration method uses the range calculated from its own samples.

In [ ]:

if "bellamy" in fit_models:
  for method, method_config in charge_methods.items():
    for config in selection_configs:
      name, label = config["name"], config["label"]
      charge_mV_ns = fit_samples[method][name]
      p0 = deepcopy(p0_bellamy)
      p0["n_total"] = [len(charge_mV_ns), 0.5 * len(charge_mV_ns), 1.5 * len(charge_mV_ns), False]

      print(f"\n===== Bellamy: {method_config['label']} - {label} =====")
      result = fit_bellamy_spe(
          charge_mV_ns, p0=p0, max_pe=maxPE, bins=nbins, fit_range=fit_ranges[method]
      )
      fit_results[method][name]["bellamy"] = result
      print_fit_result_table(result)
      print(result["diagnostics"])

      title = f"Bellamy — {method_config['label']} — {label} — {voltage_V:g} V"
      fig, ax_fit, ax_resid, ax_corr = plot_fit_summary(
          result, title=title, shrink_colorbar=0.7,
          component_visibility_fraction=0, logscale=True,
          show_event_fractions=True, max_fraction_pe=maxPE,
      )
      save_plot(fig, save_plots, save_dir, file_nickname,
                f"fit_Bellamy_{method}_{name}_{nbins}bins", Nevents=None)
      save_fit_results(result, output_path=f"{savefit_path}/{file_nickname}_{method}_{name}_bellamy.yaml")
else:
    print("Skipping Bellamy fits")

# Compact side-by-side numerical comparison of the fitted parameters.
comparison_rows = []
for method, method_config in charge_methods.items():
  for config in selection_configs:
    for model, result in fit_results[method][config["name"]].items():
        comparison_rows.append({
            "integration": method_config["label"],
            "selection": config["label"],
            "model": model,
            "events": len(fit_samples[method][config["name"]]),
            "q0_mV_ns": result["parameters"].get("q0_mV_ns"),
            "q1_mV_ns": result["parameters"].get("q1_mV_ns"),
            "q1_error_mV_ns": result["errors"].get("q1_mV_ns"),
            "mu_pe": result["parameters"].get("mu_pe"),
        })

fit_comparison = pd.DataFrame(comparison_rows)
display(fit_comparison)
